In [1]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

import cell2location

In [ ]:
# Define the directory containing the sample directories
base_dir = "/scratch/svc_td_compbio/users/kiviaho/eye_retionpathy/scs_data"
os.chdir("/scratch/svc_td_compbio/users/kiviaho/eye_retionpathy")

# Load data
sc_reference_data = sc.read_h5ad('./scs_data_clustered_annotated_250611_cellular_subtypes_for_cell2location.h5ad')
spatial_data = sc.read_h5ad('./spatial_data/aggregate_oir_spatial_data_251008.h5ad')

# Get the overlap
genes_intersection = sc_reference_data.var_names.intersection(spatial_data.var_names)
sc_reference_data = sc_reference_data[:,genes_intersection]
spatial_data = spatial_data[:,genes_intersection]

In [ ]:
sc_reference_data

In [ ]:
# prepare anndata for the regression model
cell2location.models.RegressionModel.setup_anndata(adata=sc_reference_data,
                        # 10X reaction / sample / batch
                        batch_key='sample',
                        # cell type, covariate used for constructing signatures
                        labels_key='celltype_leiden',
                       )

In [ ]:
# create the regression model
from cell2location.models import RegressionModel
mod = RegressionModel(sc_reference_data)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

In [ ]:
mod.train(max_epochs=250)

In [ ]:
mod.plot_history(20)

In [ ]:
results_folder = './c2l_dir'

# create paths and names to results folders for reference regression and cell2location models
ref_run_name = f'{results_folder}/reference_signatures'
run_name = f'{results_folder}/cell2location_map'


# In this section, we export the estimated cell abundance (summary of the posterior distribution).
sc_reference_data = mod.export_posterior(
    sc_reference_data, sample_kwargs={'num_samples': 1000, 'batch_size': 2500}
)

# Save model
mod.save(f"{ref_run_name}", overwrite=True)

# Save anndata object with results
adata_file = f"{ref_run_name}/sc.h5ad"
sc_reference_data.write(adata_file)

In [ ]:
#adata_ref = sc_reference_data
#adata_ref = sc.read_h5ad('./c2l_dir/reference_signatures/sc.h5ad')
adata_ref = mod.export_posterior(
    adata_ref, use_quantiles=True,
    # choose quantiles
    add_to_varm=["q05","q50", "q95", "q0001"],
    sample_kwargs={'batch_size': 2500}
)

In [ ]:
mod.plot_QC(summary_name="q50")

In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in adata_ref.varm.keys():
    inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
else:
    inf_aver = adata_ref.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:5]

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(spatial_data.var_names, inf_aver.index)
spatial_data = spatial_data[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

# prepare anndata for cell2location model
cell2location.models.Cell2location.setup_anndata(adata=spatial_data, batch_key="sample")

In [ ]:
# create and train the model
mod = cell2location.models.Cell2location(
    spatial_data, cell_state_df=inf_aver,
    # the expected average cell abundance: tissue-dependent
    # hyper-prior which can be estimated from paired histology:
    N_cells_per_location=30,
    # hyperparameter controlling normalisation of
    # within-experiment variation in RNA detection:
    detection_alpha=20
)
mod.view_anndata_setup()

In [ ]:
mod.train(max_epochs=20000,
          # train using full data (batch_size=None)
          batch_size=None,
          # use all data points in training because
          # we need to estimate cell abundance at all locations
          train_size=1,
          #use_gpu=True,
         )

# plot ELBO loss history during training, removing first 100 epochs from the plot
mod.plot_history(1000)
plt.legend(labels=['full data training'])

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution).
adata_vis = mod.export_posterior(
    spatial_data, sample_kwargs={'num_samples': 1000, 'batch_size': mod.adata.n_obs}
)

# Save model
mod.save(f"{run_name}", overwrite=True)

# mod = cell2location.models.Cell2location.load(f"{run_name}", spatial_data)

# Save anndata object with results
adata_file = f"{run_name}/sp.h5ad"
spatial_data.write(adata_file)
adata_file